# Data Cleaning Only — PayGo Solar Collections Portfolio

**Scope of this notebook:** clean, standardise, audit, and seal the five datasets.

This notebook intentionally **does not do feature engineering or modelling**.

## Time split policy

| Layer | Months | Purpose |
|---|---|---|
| Estimation | Oct 2024–Mar 2026 | Future model fitting / collection-curve estimation |
| Validation | Apr–Jun 2026 | Future model tuning / backtesting in the pilot era |
| Sealed test | Jul–Sep 2026 | Do not inspect until the final forecast is locked |

### Important distinction — split early, clean consistently

The safest workflow is a **hybrid**:

1. keep an immutable copy of every raw file;
2. do only the minimum schema normalisation needed to identify records correctly (column names, `contract_id` → `contractid`, ID type, and date parsing);
3. assign each record to estimation / validation / sealed-test by date;
4. apply the **same deterministic cleaning functions** to every partition;
5. fit any data-dependent transformation later using estimation data only, then carry that learned rule into validation and test.

Examples of deterministic cleaning that can be applied consistently everywhere are trimming whitespace, parsing a date, standardising a known business label, or removing a provable exact duplicate.

Examples that must **not** be learned from the sealed test are imputation values, outlier cut-offs, feature selection, category pooling based on frequency, segmentation depth, or model tuning.

This notebook never prints or summarises Jul–Sep payment outcomes while the test remains sealed.

## Dataset flaws explicitly handled

- missing values in optional demographic fields;
- `contractid` versus `contract_id`;
- forecast months embedded in source files;
- mixed / text / Excel-serial dates;
- messy service-ticket free text;
- exact duplicate rows;
- conflicting duplicate contract-month payment records;
- zero and negative payment rows;
- events before contract sale month;
- payments after a contract appears fully paid;
- orphan IDs across files;
- structurally limited outreach data;
- possible partial / thin months, with sealed-period diagnostics disabled by default.

All non-trivial corrections preserve the original value or create an audit file.

## 1. Upload the five original CSVs

Select:

- `contracts.csv`
- `payments.csv`
- `calls.csv`
- `service_tickets.csv`
- `collections_outreach.csv`

The notebook will later create downloadable ZIP files.

**Colab cannot force Chrome to save directly to Desktop.** `files.download(...)` triggers the browser download; choose Desktop if your browser asks where to save.

In [ ]:
from google.colab import files
uploaded = files.upload()

print("Uploaded files:")
for name in uploaded:
    print(" -", name)

## 2. Imports and cleaning configuration

In [ ]:
import io
import re
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

EXPECTED_DATA_START = pd.Timestamp("2024-10-01")
ESTIMATION_END = pd.Timestamp("2026-03-31")
VALIDATION_END = pd.Timestamp("2026-06-30")
SEALED_TEST_END = pd.Timestamp("2026-09-30")

OUTPUT_ROOT = Path("/content/cleaning_outputs")
DEV_DIR = OUTPUT_ROOT / "development_through_jun_2026"
SEALED_DIR = OUTPUT_ROOT / "SEALED_TEST_JUL_SEP_2026"
AUDIT_DIR = OUTPUT_ROOT / "audit"
FULL_DIR = OUTPUT_ROOT / "cleaned_full"

for p in [DEV_DIR, SEALED_DIR, AUDIT_DIR, FULL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Keep this FALSE while cleaning / building the future model.
# Turn TRUE only after the Jul-Sep forecast is locked.
UNSEAL_FINAL_TEST_DIAGNOSTICS = False

## 3. Robust file loading

In [ ]:
def normalise_filename(name):
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")

def find_uploaded_file(keyword):
    key = normalise_filename(keyword)
    matches = [
        name for name in uploaded.keys()
        if key in normalise_filename(name)
    ]
    if not matches:
        raise FileNotFoundError(f"Could not find uploaded file matching: {keyword}")
    if len(matches) > 1:
        print(f"Multiple matches for {keyword}: {matches}; using {matches[0]}")
    return matches[0]

def read_uploaded_csv(keyword):
    filename = find_uploaded_file(keyword)
    return pd.read_csv(io.BytesIO(uploaded[filename]))

contracts_raw = read_uploaded_csv("contracts")
payments_raw = read_uploaded_csv("payments")
calls_raw = read_uploaded_csv("calls")
service_raw = read_uploaded_csv("service_tickets")
outreach_raw = read_uploaded_csv("collections_outreach")

raw_shapes = pd.DataFrame({
    "dataset": [
        "contracts", "payments", "calls",
        "service_tickets", "collections_outreach"
    ],
    "rows": [
        len(contracts_raw), len(payments_raw), len(calls_raw),
        len(service_raw), len(outreach_raw)
    ],
    "columns": [
        contracts_raw.shape[1], payments_raw.shape[1], calls_raw.shape[1],
        service_raw.shape[1], outreach_raw.shape[1]
    ]
})

display(raw_shapes)

## 4. Cleaning helpers

In [ ]:
def clean_column_names(df):
    out = df.copy()
    out.columns = (
        out.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
    )
    return out.dropna(axis=1, how="all")

def clean_id(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    )

def clean_text(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    )

def parse_mixed_date(series):
    raw = series.astype("string").str.strip()
    numeric = pd.to_numeric(raw, errors="coerce")

    result = pd.to_datetime(
        raw.where(numeric.isna()),
        errors="coerce"
    )

    numeric_mask = numeric.notna()
    if numeric_mask.any():
        result.loc[numeric_mask] = pd.to_datetime(
            numeric.loc[numeric_mask],
            unit="D",
            origin="1899-12-30",
            errors="coerce"
        )
    return result

def to_month_end(series):
    dates = pd.to_datetime(series, errors="coerce")
    return dates.dt.to_period("M").dt.to_timestamp("M")

def assign_time_layer(date_series):
    d = pd.to_datetime(date_series, errors="coerce")
    layer = pd.Series(pd.NA, index=d.index, dtype="string")

    layer.loc[d <= ESTIMATION_END] = "estimation"
    layer.loc[
        (d > ESTIMATION_END) & (d <= VALIDATION_END)
    ] = "validation"
    layer.loc[
        (d > VALIDATION_END) & (d <= SEALED_TEST_END)
    ] = "sealed_test"

    layer.loc[d > SEALED_TEST_END] = "post_test"
    return layer

def collapse_repeated_phrase(text):
    if pd.isna(text):
        return text

    s = re.sub(r"\s+", " ", str(text).strip())

    # If a complete phrase has simply been repeated, collapse it.
    words = s.split()
    n = len(words)

    for chunk_len in range(1, n // 2 + 1):
        if n % chunk_len != 0:
            continue
        chunk = words[:chunk_len]
        repeated = chunk * (n // chunk_len)
        if repeated == words:
            return " ".join(chunk)

    return s

## 5. Contracts cleaning

### Policy for optional demographics

`customer_gender`, `household_size`, and `occupation` are explicitly “if given” fields. Missing values are therefore **preserved as missing**. This notebook does not impute them and does not create an `UNKNOWN` modelling category.

### Additional confirmed issue

Some financed contracts have `perc_deposit > 1`. Where this occurs and `price_usd > 0`, preserve the original value and convert the apparent deposit amount to a proportion using:

`corrected perc_deposit = original perc_deposit / price_usd`

Every corrected record is flagged.

In [ ]:
contracts = clean_column_names(contracts_raw)

contracts["contractid"] = clean_id(contracts["contractid"])
contracts["sales_person_id"] = clean_id(contracts["sales_person_id"])

contracts["sales_month_original"] = contracts["sales_month"]
contracts["sales_month"] = to_month_end(
    parse_mixed_date(contracts["sales_month"])
)

numeric_cols = [
    "household_size",
    "price_usd",
    "perc_deposit",
    "daily_amount_usd",
    "tenor_length"
]

for col in numeric_cols:
    contracts[col] = pd.to_numeric(
        contracts[col],
        errors="coerce"
    )

contracts["region"] = clean_text(contracts["region"]).str.title()
contracts["customer_gender"] = clean_text(contracts["customer_gender"]).str.upper()
contracts["occupation"] = clean_text(contracts["occupation"]).str.upper()
contracts["contract_type"] = clean_text(contracts["contract_type"]).str.upper()
contracts["payment_frequency"] = clean_text(contracts["payment_frequency"]).str.upper()
contracts["product"] = clean_text(contracts["product"])

contracts["perc_deposit_original"] = contracts["perc_deposit"]

deposit_scale_mask = (
    contracts["contract_type"].eq("FINANCED")
    & contracts["perc_deposit"].gt(1)
    & contracts["price_usd"].gt(0)
)

contracts["deposit_scale_corrected"] = deposit_scale_mask

contracts.loc[deposit_scale_mask, "perc_deposit"] = (
    contracts.loc[deposit_scale_mask, "perc_deposit"]
    / contracts.loc[deposit_scale_mask, "price_usd"]
)

contracts["time_layer"] = assign_time_layer(
    contracts["sales_month"]
)

contract_missingness = (
    contracts.isna()
    .sum()
    .rename("missing")
    .to_frame()
)

contract_key_checks = pd.DataFrame({
    "check": [
        "rows",
        "missing_contractid",
        "duplicate_contractid_rows",
        "invalid_sales_month",
        "deposit_scale_corrected",
        "financed_deposit_outside_0_1_after_fix",
        "cash_contract_perc_deposit_not_1"
    ],
    "value": [
        len(contracts),
        int(contracts["contractid"].isna().sum()),
        int(contracts["contractid"].duplicated(keep=False).sum()),
        int(contracts["sales_month"].isna().sum()),
        int(deposit_scale_mask.sum()),
        int((
            contracts.loc[
                contracts["contract_type"].eq("FINANCED"),
                "perc_deposit"
            ].notna()
            &
            ~contracts.loc[
                contracts["contract_type"].eq("FINANCED"),
                "perc_deposit"
            ].between(0, 1)
        ).sum()),
        int((
            contracts.loc[
                contracts["contract_type"].eq("CASH"),
                "perc_deposit"
            ].notna()
            &
            ~np.isclose(
                contracts.loc[
                    contracts["contract_type"].eq("CASH"),
                    "perc_deposit"
                ],
                1.0
            )
        ).sum())
    ]
})

display(contract_missingness)
display(contract_key_checks)

## 6. Payments cleaning

### Payment duplicate policy

The data dictionary says `total_paid` is the **total amount paid in that month**. Therefore:

- exact duplicate rows are safe to remove;
- two rows for the same `contractid + pay_month` with **different** amounts are not automatically summed, averaged, or overwritten;
- conflicting contract-month records are quarantined in an audit file for review.

This avoids accidentally double-counting monthly collections.

### Zero and negative payments

They are flagged, not silently deleted:

- `0` may represent an empty monthly record;
- `< 0` may represent a reversal or correction.

Their business meaning should be confirmed before deciding whether they belong in the future cash model.

In [ ]:
payments = clean_column_names(payments_raw)

# Critical join-key repair.
if "contract_id" in payments.columns:
    payments = payments.rename(
        columns={"contract_id": "contractid"}
    )

payments["contractid"] = clean_id(
    payments["contractid"]
)

payments["pay_month_original"] = payments["pay_month"]
payments["pay_month"] = to_month_end(
    parse_mixed_date(payments["pay_month"])
)

payments["total_paid"] = pd.to_numeric(
    payments["total_paid"],
    errors="coerce"
)

payments["time_layer"] = assign_time_layer(
    payments["pay_month"]
)

# Rows before the stated analysis window are retained for audit, not silently used.
payments["before_expected_data_start"] = (
    payments["pay_month"].notna()
    & (payments["pay_month"] < EXPECTED_DATA_START)
)

# Exact duplicates: preserve audit, then remove.
payment_exact_duplicate_mask = payments.duplicated(
    subset=["contractid", "pay_month", "total_paid"],
    keep=False
)

payment_exact_duplicates = payments.loc[
    payment_exact_duplicate_mask
].copy()

payments = payments.drop_duplicates(
    subset=["contractid", "pay_month", "total_paid"],
    keep="first"
).copy()

# Remaining same-contract same-month rows are conflicting monthly totals.
conflicting_payment_mask = payments.duplicated(
    subset=["contractid", "pay_month"],
    keep=False
)

payment_conflicting_contract_months = (
    payments.loc[conflicting_payment_mask]
    .sort_values(["contractid", "pay_month", "total_paid"])
    .copy()
)

payments["conflicting_contract_month"] = payments.index.isin(
    payment_conflicting_contract_months.index
)

payments["zero_payment"] = payments["total_paid"].eq(0)
payments["negative_payment"] = payments["total_paid"].lt(0)

payment_key_checks = pd.DataFrame({
    "check": [
        "rows_after_exact_deduplication",
        "missing_contractid",
        "invalid_pay_month",
        "missing_total_paid",
        "exact_duplicate_rows_quarantined",
        "conflicting_contract_month_rows",
        "zero_payment_rows",
        "negative_payment_rows",
        "rows_before_expected_data_start"
    ],
    "value": [
        len(payments),
        int(payments["contractid"].isna().sum()),
        int(payments["pay_month"].isna().sum()),
        int(payments["total_paid"].isna().sum()),
        len(payment_exact_duplicates),
        len(payment_conflicting_contract_months),
        int(payments["zero_payment"].sum()),
        int(payments["negative_payment"].sum()),
        int(payments["before_expected_data_start"].sum())
    ]
})

display(payment_key_checks)

## 7. Calls cleaning

Calls are event-level data. Two calls on the same day for the same reason may both be real.

Therefore duplicate-looking call rows are **flagged but not deleted** unless there is a true event identifier proving duplication.

In [ ]:
calls = clean_column_names(calls_raw)

calls["contractid"] = clean_id(calls["contractid"])

calls["call_date_original"] = calls["call_date"]
calls["call_date"] = parse_mixed_date(
    calls["call_date"]
)

calls["call_reason_original"] = calls["call_reason"]
calls["call_reason"] = (
    clean_text(calls["call_reason"])
    .str.upper()
    .str.replace(r"\s+", "_", regex=True)
)

calls["time_layer"] = assign_time_layer(
    calls["call_date"]
)

calls["possible_duplicate_event"] = calls.duplicated(
    subset=["contractid", "call_date", "call_reason"],
    keep=False
)

call_key_checks = pd.DataFrame({
    "check": [
        "rows",
        "missing_contractid",
        "invalid_call_date",
        "missing_call_reason",
        "possible_duplicate_event_rows"
    ],
    "value": [
        len(calls),
        int(calls["contractid"].isna().sum()),
        int(calls["call_date"].isna().sum()),
        int(calls["call_reason"].isna().sum()),
        int(calls["possible_duplicate_event"].sum())
    ]
})

display(call_key_checks)

## 8. Service-ticket cleaning

The original reason is preserved. Cleaning is deterministic:

1. trim whitespace;
2. replace underscores with spaces;
3. collapse a phrase repeated verbatim, e.g. `battery fault battery fault battery fault` → `battery fault`;
4. map obvious spelling / wording variants to a canonical category;
5. retain any unmapped value in a standardised form so it can be reviewed rather than lost.

### Cable taxonomy

These are deliberately kept separate because they represent different reported failure modes:

- `CABLE_CUT` — explicit physical cut / break in a cable;
- `CABLE_FAULT` — a cable-level fault where a physical cut is not stated;
- `WIRE_FAULT` — a wire fault reported as such.

The cleaning step standardises spelling and case only; it does **not** merge these three concepts.


In [ ]:
service = clean_column_names(service_raw)

service["contractid"] = clean_id(
    service["contractid"]
)

service["ticket_date_original"] = service["ticket_date"]
service["ticket_date"] = parse_mixed_date(
    service["ticket_date"]
)

service["ticket_reason_original"] = service["ticket_reason"]

reason_key = (
    clean_text(service["ticket_reason"])
    .str.lower()
    .str.replace("_", " ", regex=False)
    .map(collapse_repeated_phrase)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

ticket_reason_map = {
    "battery fault": "BATTERY_FAULT",
    "battery failure": "BATTERY_FAULT",
    "batt fault": "BATTERY_FAULT",

    "charging issue": "CHARGING_ISSUE",
    "not charging": "CHARGING_ISSUE",
    "won't charge": "CHARGING_ISSUE",
    "no charge": "CHARGING_ISSUE",

    "panel damage": "PANEL_DAMAGE",
    "panel broken": "PANEL_DAMAGE",
    "panel cracked": "PANEL_DAMAGE",

    "cable cut": "CABLE_CUT",
    "cable fault": "CABLE_FAULT",
    "wire fault": "WIRE_FAULT",

    "lamp fault": "LIGHT_FAULT",
    "bulb not working": "LIGHT_FAULT",
    "light fault": "LIGHT_FAULT",

    "other": "OTHER",
    "misc": "OTHER"
}

service["ticket_reason"] = (
    reason_key.map(ticket_reason_map)
    .fillna(
        reason_key
        .str.upper()
        .str.replace(" ", "_", regex=False)
    )
)

service["ticket_outcome_original"] = service["ticket_outcome"]
service["ticket_outcome"] = (
    clean_text(service["ticket_outcome"])
    .str.upper()
    .str.replace(r"\s+", "_", regex=True)
)

service["time_layer"] = assign_time_layer(
    service["ticket_date"]
)

ticket_reason_audit = (
    service.groupby(
        ["ticket_reason_original", "ticket_reason"],
        dropna=False
    )
    .size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
)

service_key_checks = pd.DataFrame({
    "check": [
        "rows",
        "missing_contractid",
        "invalid_ticket_date",
        "missing_ticket_reason",
        "canonical_ticket_reason_count"
    ],
    "value": [
        len(service),
        int(service["contractid"].isna().sum()),
        int(service["ticket_date"].isna().sum()),
        int(service["ticket_reason"].isna().sum()),
        int(service["ticket_reason"].nunique(dropna=True))
    ]
})

display(service_key_checks)
display(ticket_reason_audit.head(50))

## 9. Collections outreach cleaning

This table is structurally different from the other tables: it contains only the two regional pilots and starts in Apr-2026.

That is **not cleaned away**. It is preserved as an important scope limitation.

No country-wide inference or uplift is estimated in this notebook.

In [ ]:
outreach = clean_column_names(outreach_raw)

outreach["contractid"] = clean_id(
    outreach["contractid"]
)

outreach["contact_month_original"] = outreach["contact_month"]
outreach["contact_month"] = to_month_end(
    parse_mixed_date(outreach["contact_month"])
)

outreach["region"] = clean_text(
    outreach["region"]
).str.title()

outreach["channel"] = clean_text(
    outreach["channel"]
).str.upper()

outreach["attempts"] = pd.to_numeric(
    outreach["attempts"],
    errors="coerce"
)

outreach["cost_usd"] = pd.to_numeric(
    outreach["cost_usd"],
    errors="coerce"
)

outreach["reached_original"] = outreach["reached"]

outreach["reached"] = (
    outreach["reached"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False
    })
    .astype("boolean")
)

outreach["time_layer"] = assign_time_layer(
    outreach["contact_month"]
)

outreach["possible_duplicate_contract_month"] = outreach.duplicated(
    subset=["contractid", "contact_month"],
    keep=False
)

outreach_key_checks = pd.DataFrame({
    "check": [
        "rows",
        "missing_contractid",
        "invalid_contact_month",
        "invalid_attempts",
        "invalid_cost_usd",
        "invalid_reached",
        "duplicate_contract_month_rows"
    ],
    "value": [
        len(outreach),
        int(outreach["contractid"].isna().sum()),
        int(outreach["contact_month"].isna().sum()),
        int(outreach["attempts"].isna().sum()),
        int(outreach["cost_usd"].isna().sum()),
        int(outreach["reached"].isna().sum()),
        int(outreach["possible_duplicate_contract_month"].sum())
    ]
})

display(outreach_key_checks)

print("Pilot structure (safe structural check):")
display(
    outreach.groupby(["region", "channel"])
    .size()
    .to_frame("rows")
)

## 10. Cross-file join integrity and orphan IDs

All IDs are compared to cleaned `contracts.contractid`.

Orphans are **flagged and exported**, not silently discarded.

In [ ]:
contract_id_set = set(
    contracts["contractid"].dropna()
)

def add_orphan_flag(df):
    out = df.copy()
    out["orphan_contractid"] = (
        out["contractid"].notna()
        &
        ~out["contractid"].isin(contract_id_set)
    )
    return out

payments = add_orphan_flag(payments)
calls = add_orphan_flag(calls)
service = add_orphan_flag(service)
outreach = add_orphan_flag(outreach)

orphan_summary = pd.DataFrame({
    "dataset": [
        "payments", "calls",
        "service_tickets", "collections_outreach"
    ],
    "orphan_rows": [
        int(payments["orphan_contractid"].sum()),
        int(calls["orphan_contractid"].sum()),
        int(service["orphan_contractid"].sum()),
        int(outreach["orphan_contractid"].sum())
    ],
    "orphan_unique_ids": [
        payments.loc[payments["orphan_contractid"], "contractid"].nunique(),
        calls.loc[calls["orphan_contractid"], "contractid"].nunique(),
        service.loc[service["orphan_contractid"], "contractid"].nunique(),
        outreach.loc[outreach["orphan_contractid"], "contractid"].nunique()
    ]
})

display(orphan_summary)

## 11. Events before sale month

Because `sales_month` is only month-level, same-month events are allowed.

An event is flagged only if it occurs in a calendar month **strictly before** the contract's sale month.

Nothing is deleted automatically.

In [ ]:
contract_dates = contracts[
    ["contractid", "sales_month"]
].copy()

def flag_before_sale(df, date_col):
    out = df.merge(
        contract_dates,
        on="contractid",
        how="left",
        validate="m:1"
    )

    event_period = pd.to_datetime(
        out[date_col],
        errors="coerce"
    ).dt.to_period("M")

    sale_period = pd.to_datetime(
        out["sales_month"],
        errors="coerce"
    ).dt.to_period("M")

    out["before_sale_month"] = (
        event_period < sale_period
    )

    return out

payments = flag_before_sale(payments, "pay_month")
calls = flag_before_sale(calls, "call_date")
service = flag_before_sale(service, "ticket_date")
outreach = flag_before_sale(outreach, "contact_month")

before_sale_summary = pd.DataFrame({
    "dataset": [
        "payments", "calls",
        "service_tickets", "collections_outreach"
    ],
    "rows_before_sale_month": [
        int(payments["before_sale_month"].sum()),
        int(calls["before_sale_month"].sum()),
        int(service["before_sale_month"].sum()),
        int(outreach["before_sale_month"].sum())
    ]
})

display(before_sale_summary)

## 12. Payments after the contract appears fully paid

This is an **audit flag**, not a deletion rule.

A payment is flagged when the cumulative **prior positive payments** are already at or above the contract price.

Why positive payments only? Negative rows may be reversals/corrections, and their business meaning is not yet confirmed. The audit therefore identifies suspicious post-payoff cash without pretending to resolve reversals.

In [ ]:
payment_payoff = payments.merge(
    contracts[["contractid", "price_usd"]],
    on="contractid",
    how="left",
    validate="m:1"
)

payment_payoff = payment_payoff.sort_values(
    ["contractid", "pay_month"]
).copy()

payment_payoff["_positive_paid"] = (
    payment_payoff["total_paid"]
    .clip(lower=0)
    .fillna(0)
)

payment_payoff["cum_positive_before"] = (
    payment_payoff.groupby("contractid")["_positive_paid"]
    .cumsum()
    - payment_payoff["_positive_paid"]
)

payment_payoff["payment_after_apparent_payoff"] = (
    payment_payoff["price_usd"].notna()
    &
    (payment_payoff["cum_positive_before"] >= payment_payoff["price_usd"])
    &
    (payment_payoff["_positive_paid"] > 0)
)

payments = payments.merge(
    payment_payoff[
        ["contractid", "pay_month", "total_paid",
         "payment_after_apparent_payoff"]
    ],
    on=["contractid", "pay_month", "total_paid"],
    how="left"
)

payments["payment_after_apparent_payoff"] = (
    payments["payment_after_apparent_payoff"]
    .fillna(False)
)

print(
    "Payments after apparent full payoff:",
    int(payments["payment_after_apparent_payoff"].sum())
)

## 13. Partial / thin month diagnostic — DEVELOPMENT PERIOD ONLY

This is an **extraction-completeness diagnostic**, not a rule for deleting low-performing months.

A genuinely weak collections month can have lower dollars while still containing a normal number of payment rows and paying contracts. A truncated / partial extract usually shows several volume signals collapsing together.

For each development month the notebook therefore compares:

- total cash collected;
- number of payment rows;
- number of unique paying contracts;
- mean payment per paying contract;

against the previous 3-month median.

Interpret the measures together:

- **cash falls, but rows and unique contracts remain near normal** → more consistent with genuinely smaller payments;
- **cash, rows and unique contracts all collapse** → possible incomplete month / extraction issue;
- **a month occurs before the stated Oct-2024 start** → treat it as a pre-period anomaly and audit separately.

No month is automatically removed from modelling by this diagnostic.


In [ ]:
payments_dev_for_diagnostic = payments[
    payments["pay_month"] <= VALIDATION_END
].copy()

monthly_payment_diagnostic = (
    payments_dev_for_diagnostic
    .groupby("pay_month", as_index=False)
    .agg(
        total_paid=("total_paid", "sum"),
        rows=("contractid", "size"),
        unique_contracts=("contractid", "nunique")
    )
    .sort_values("pay_month")
)

monthly_payment_diagnostic["mean_paid_per_contract"] = (
    monthly_payment_diagnostic["total_paid"]
    / monthly_payment_diagnostic["unique_contracts"].replace(0, np.nan)
)

monthly_payment_diagnostic["before_expected_data_start"] = (
    monthly_payment_diagnostic["pay_month"] < EXPECTED_DATA_START
)

for col in ["total_paid", "rows", "unique_contracts", "mean_paid_per_contract"]:
    monthly_payment_diagnostic[
        f"{col}_prev3_median"
    ] = (
        monthly_payment_diagnostic[col]
        .shift(1)
        .rolling(3)
        .median()
    )

    monthly_payment_diagnostic[
        f"{col}_ratio_to_prev3"
    ] = (
        monthly_payment_diagnostic[col]
        /
        monthly_payment_diagnostic[
            f"{col}_prev3_median"
        ]
    )

display(monthly_payment_diagnostic)

## 14. OPTIONAL sealed-period completeness diagnostic

Keep this disabled while preparing the forecast.

After the forecast is locked, set:

`UNSEAL_FINAL_TEST_DIAGNOSTICS = True`

This will reveal Jul–Sep payment totals / row counts and help determine whether the final test months are complete.

In [ ]:
if UNSEAL_FINAL_TEST_DIAGNOSTICS:
    sealed_payment_diagnostic = (
        payments[
            (payments["pay_month"] > VALIDATION_END)
            &
            (payments["pay_month"] <= SEALED_TEST_END)
        ]
        .groupby("pay_month", as_index=False)
        .agg(
            total_paid=("total_paid", "sum"),
            rows=("contractid", "size"),
            unique_contracts=("contractid", "nunique")
        )
        .sort_values("pay_month")
    )

    display(sealed_payment_diagnostic)
else:
    print(
        "SEALED: Jul-Sep payment totals are not displayed. "
        "Leave UNSEAL_FINAL_TEST_DIAGNOSTICS = False until the forecast is locked."
    )

## 15. Create clean development and sealed files

### Important

For `contracts.csv`, only **new contracts sold Jul–Sep** are sealed. Contracts sold before Jul remain in development because their contract terms would have been known on 30-Jun-2026.

For event / payment tables, Jul–Sep events are routed to the sealed files based on their event month.

In [ ]:
def split_dev_sealed(df, date_col):
    dev = df[
        pd.to_datetime(df[date_col], errors="coerce")
        <= VALIDATION_END
    ].copy()

    sealed = df[
        (
            pd.to_datetime(df[date_col], errors="coerce")
            > VALIDATION_END
        )
        &
        (
            pd.to_datetime(df[date_col], errors="coerce")
            <= SEALED_TEST_END
        )
    ].copy()

    return dev, sealed

contracts_dev, contracts_sealed = split_dev_sealed(
    contracts,
    "sales_month"
)

payments_dev, payments_sealed = split_dev_sealed(
    payments,
    "pay_month"
)

calls_dev, calls_sealed = split_dev_sealed(
    calls,
    "call_date"
)

service_dev, service_sealed = split_dev_sealed(
    service,
    "ticket_date"
)

outreach_dev, outreach_sealed = split_dev_sealed(
    outreach,
    "contact_month"
)

# Development files: safe working set through Jun-2026.
contracts_dev.to_csv(
    DEV_DIR / "contracts_clean_through_jun_2026.csv",
    index=False
)
payments_dev.to_csv(
    DEV_DIR / "payments_clean_through_jun_2026.csv",
    index=False
)
calls_dev.to_csv(
    DEV_DIR / "calls_clean_through_jun_2026.csv",
    index=False
)
service_dev.to_csv(
    DEV_DIR / "service_tickets_clean_through_jun_2026.csv",
    index=False
)
outreach_dev.to_csv(
    DEV_DIR / "collections_outreach_clean_through_jun_2026.csv",
    index=False
)

# Sealed files: do not open until final forecast is locked.
contracts_sealed.to_csv(
    SEALED_DIR / "SEALED_new_contracts_jul_sep_2026.csv",
    index=False
)
payments_sealed.to_csv(
    SEALED_DIR / "SEALED_payments_jul_sep_2026.csv",
    index=False
)
calls_sealed.to_csv(
    SEALED_DIR / "SEALED_calls_jul_sep_2026.csv",
    index=False
)
service_sealed.to_csv(
    SEALED_DIR / "SEALED_service_tickets_jul_sep_2026.csv",
    index=False
)
outreach_sealed.to_csv(
    SEALED_DIR / "SEALED_collections_outreach_jul_sep_2026.csv",
    index=False
)

print("Development files written through 30-Jun-2026.")
print("Jul-Sep files written to separate SEALED directory.")

## 16. Save full cleaned files

These exist for reproducibility, but for the forecasting work use only the **development ZIP** until the final forecast is locked.

In [ ]:
contracts.to_csv(
    FULL_DIR / "contracts_clean_full.csv",
    index=False
)
payments.to_csv(
    FULL_DIR / "payments_clean_full.csv",
    index=False
)
calls.to_csv(
    FULL_DIR / "calls_clean_full.csv",
    index=False
)
service.to_csv(
    FULL_DIR / "service_tickets_clean_full.csv",
    index=False
)
outreach.to_csv(
    FULL_DIR / "collections_outreach_clean_full.csv",
    index=False
)

print("Full cleaned copies written for reproducibility.")

## 17. Save audit files

In [ ]:
# Key audits
payment_exact_duplicates.to_csv(
    AUDIT_DIR / "payment_exact_duplicates_removed.csv",
    index=False
)

payment_conflicting_contract_months.to_csv(
    AUDIT_DIR / "payment_conflicting_contract_months_REVIEW.csv",
    index=False
)

calls[
    calls["possible_duplicate_event"]
].to_csv(
    AUDIT_DIR / "calls_possible_duplicate_events_REVIEW.csv",
    index=False
)

ticket_reason_audit.to_csv(
    AUDIT_DIR / "ticket_reason_mapping_audit.csv",
    index=False
)

payments[
    payments["orphan_contractid"]
].to_csv(
    AUDIT_DIR / "orphan_payments.csv",
    index=False
)

calls[
    calls["orphan_contractid"]
].to_csv(
    AUDIT_DIR / "orphan_calls.csv",
    index=False
)

service[
    service["orphan_contractid"]
].to_csv(
    AUDIT_DIR / "orphan_service_tickets.csv",
    index=False
)

outreach[
    outreach["orphan_contractid"]
].to_csv(
    AUDIT_DIR / "orphan_collections_outreach.csv",
    index=False
)

payments[
    payments["before_sale_month"]
].to_csv(
    AUDIT_DIR / "payments_before_sale_REVIEW.csv",
    index=False
)

calls[
    calls["before_sale_month"]
].to_csv(
    AUDIT_DIR / "calls_before_sale_REVIEW.csv",
    index=False
)

service[
    service["before_sale_month"]
].to_csv(
    AUDIT_DIR / "service_tickets_before_sale_REVIEW.csv",
    index=False
)

outreach[
    outreach["before_sale_month"]
].to_csv(
    AUDIT_DIR / "outreach_before_sale_REVIEW.csv",
    index=False
)

payments[
    payments["zero_payment"]
].to_csv(
    AUDIT_DIR / "zero_payment_rows_REVIEW.csv",
    index=False
)

payments[
    payments["negative_payment"]
].to_csv(
    AUDIT_DIR / "negative_payment_rows_REVIEW.csv",
    index=False
)

payments[
    payments["payment_after_apparent_payoff"]
].to_csv(
    AUDIT_DIR / "payments_after_apparent_payoff_REVIEW.csv",
    index=False
)

contracts[
    contracts["deposit_scale_corrected"]
].to_csv(
    AUDIT_DIR / "contracts_deposit_scale_corrected.csv",
    index=False
)


payments[
    payments["before_expected_data_start"]
].to_csv(
    AUDIT_DIR / "payments_before_expected_oct_2024_start_REVIEW.csv",
    index=False
)

monthly_payment_diagnostic.to_csv(
    AUDIT_DIR / "monthly_payment_completeness_through_jun_2026.csv",
    index=False
)

print("Audit files saved.")

## 18. Cleaning summary report

**Duplicate-count note:** `payments_exact_duplicate_rows_quarantined` counts every row participating in an exact-duplicate group. The number physically removed is smaller because one copy is retained. For example, a duplicate pair contributes 2 rows to the audit but only 1 removed row.


In [ ]:
cleaning_summary = pd.DataFrame([
    ["contracts_rows", len(contracts)],
    ["contracts_optional_gender_missing", int(contracts["customer_gender"].isna().sum())],
    ["contracts_optional_household_size_missing", int(contracts["household_size"].isna().sum())],
    ["contracts_optional_occupation_missing", int(contracts["occupation"].isna().sum())],
    ["contracts_deposit_scale_corrected", int(contracts["deposit_scale_corrected"].sum())],

    ["payments_rows_after_exact_dedup", len(payments)],
    ["payments_exact_duplicate_rows_quarantined", len(payment_exact_duplicates)],
    ["payments_conflicting_contract_month_rows", len(payment_conflicting_contract_months)],
    ["payments_zero_rows", int(payments["zero_payment"].sum())],
    ["payments_negative_rows", int(payments["negative_payment"].sum())],
    ["payments_before_expected_oct_2024_start", int(payments["before_expected_data_start"].sum())],
    ["payments_before_sale_rows", int(payments["before_sale_month"].sum())],
    ["payments_after_apparent_payoff_rows", int(payments["payment_after_apparent_payoff"].sum())],

    ["calls_possible_duplicate_event_rows", int(calls["possible_duplicate_event"].sum())],
    ["calls_before_sale_rows", int(calls["before_sale_month"].sum())],

    ["service_ticket_canonical_reason_count", int(service["ticket_reason"].nunique(dropna=True))],
    ["service_before_sale_rows", int(service["before_sale_month"].sum())],

    ["outreach_before_sale_rows", int(outreach["before_sale_month"].sum())],
    ["outreach_regions", ", ".join(sorted(outreach["region"].dropna().astype(str).unique()))],
    ["outreach_channels", ", ".join(sorted(outreach["channel"].dropna().astype(str).unique()))],

    ["orphan_payment_rows", int(payments["orphan_contractid"].sum())],
    ["orphan_call_rows", int(calls["orphan_contractid"].sum())],
    ["orphan_service_rows", int(service["orphan_contractid"].sum())],
    ["orphan_outreach_rows", int(outreach["orphan_contractid"].sum())],
], columns=["check", "value"])

cleaning_summary.to_csv(
    AUDIT_DIR / "cleaning_summary.csv",
    index=False
)

display(cleaning_summary)

## 19. Pack outputs into ZIP files

You will get three separate ZIPs:

1. **development_through_jun_2026.zip** — this is the file set to use for future feature engineering and model development.
2. **data_quality_audit.zip** — review / evidence for cleaning decisions.
3. **SEALED_TEST_JUL_SEP_2026.zip** — do not open until the final forecast is locked.

A full cleaned-data ZIP is also produced for reproducibility, but keep it away from the modelling workflow to reduce accidental leakage.

In [ ]:
def zip_directory(source_dir, zip_path):
    with zipfile.ZipFile(
        zip_path,
        "w",
        zipfile.ZIP_DEFLATED
    ) as z:
        for path in source_dir.rglob("*"):
            if path.is_file():
                z.write(
                    path,
                    arcname=path.relative_to(source_dir)
                )

DEV_ZIP = Path("/content/development_through_jun_2026.zip")
AUDIT_ZIP = Path("/content/data_quality_audit.zip")
SEALED_ZIP = Path("/content/SEALED_TEST_JUL_SEP_2026_DO_NOT_OPEN.zip")
FULL_ZIP = Path("/content/cleaned_full_reproducibility_only.zip")

zip_directory(DEV_DIR, DEV_ZIP)
zip_directory(AUDIT_DIR, AUDIT_ZIP)
zip_directory(SEALED_DIR, SEALED_ZIP)
zip_directory(FULL_DIR, FULL_ZIP)

print("Created:")
print(DEV_ZIP)
print(AUDIT_ZIP)
print(SEALED_ZIP)
print(FULL_ZIP)

## 20. Download buttons

Run this cell and save the files to Desktop if your browser asks where to save them.

In [ ]:
from google.colab import files

files.download(str(DEV_ZIP))
files.download(str(AUDIT_ZIP))
files.download(str(SEALED_ZIP))
files.download(str(FULL_ZIP))

# Stop here

At this point the task is **cleaning only**.

Do not create lags, months-on-book, collection curves, arrears states, rolling payment rates, call counts, service-ticket indicators, outreach features, or any other model inputs yet.

Those are feature-engineering choices and should be discussed separately after the cleaned development files and audits are reviewed.